# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides an example for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will learn how to load metadata, investigate record sets and fields (by their `@id`), extract and process data, and apply exploratory analysis, all while referencing entities via their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. The primary metadata object provides details such as title, description, version, etc.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata (as an object, not dict!)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")

## 2. Data Overview

Review available record sets and their details. Every record set, field, or column should be referenced by its Croissant `@id`. 

Below, we extract and print the available record sets (with their `@id`) defined in the dataset, including the fields each record set contains (with their `@id` and labels, if available).

In [ ]:
# List available record sets by their `@id` and list their fields (with IDs)
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata. This often indicates a pure tabular (single CSV) or simple dataset.")

else:
    print("Available record sets (with @id):\n")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                label = getattr(field, 'label', None) or getattr(field, 'name', None)
                print(f"    - {field.id} ({label})")
        else:
            print("  No fields found in this set.")

## 3. Data Extraction

Load data from each available record set (using their `@id`) into pandas DataFrames. Use variables to reference record sets and fields by their Croissant `@id`.

If the dataset does not define explicit record sets, load all records using the dataset default.

In [ ]:
# Dynamically extract and store data from each record set (referenced by `@id`)
dataframes = dict()

if not dataset.record_sets:
    # If no explicit record sets, try to load everything under the default
    print("No explicit record sets found. Attempting to load records via the dataset default...")
    records = list(dataset.records())
    if records:
        default_record_set_id = 'default'
        dataframes[default_record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records into DataFrame for record set @id='{default_record_set_id}'.")
        print(f"Columns: {dataframes[default_record_set_id].columns.tolist()}")
        display(dataframes[default_record_set_id].head())
    else:
        print("No records loaded.")
else:
    record_set_ids = [rs.id for rs in dataset.record_sets]
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records into DataFrame for record set @id='{rs_id}'.")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for record set @id='{rs_id}'.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, grouping etc. All references below are made using `@id` variables as found above.

*Note: Replace `<record_set_id>` and field IDs below with those available in your DataFrame(s).*


In [ ]:
# --- Configure analysis for your dataset: set these @ids to available ones ---
if dataframes:
    # Take the first available DataFrame
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Analyzing record set: {main_record_set_id}")
    
    # Suggest a numeric field by looking for likely numeric columns
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use first numeric column
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric fields detected for EDA. Skipping numeric filtering.")
        numeric_field_id = None
        
    # Suggest a group field if available
    group_candidates = [col for col in df.columns if df[col].nunique() < max(10, 0.05*len(df))]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        print(f"Grouping by: {group_field_id}")

    if numeric_field_id:
        # Filter records with numeric_field > threshold (choose a quantile for dataset-specific threshold)
        quant = 0.9
        threshold = df[numeric_field_id].quantile(quant)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\\nFiltered records where '{numeric_field_id}' > {threshold:.3f} (90th percentile):\n")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group (if possible and meaningful)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
            print(f"\\nGrouped by '{group_field_id}', mean of '{numeric_field_id}':")
            display(grouped_df)
else:
    print("No dataframes found from data extraction step to analyze.")

## 5. Visualization

Visualize data distributions or relationships between fields, such as histograms of numeric fields, bar plots of grouped means, or scatter plots between two variables.

*Note: Plots are based on column `@id` names derived above.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    # Histogram for numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Insufficient data or numeric field not found for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` Python library, referencing all key entities (`record sets`, `fields`, and `columns`) by their Croissant `@id`. 

- We loaded the dataset, explored its metadata, and dynamically listed record sets and their contents.
- We extracted and visualized data using pandas and seaborn.
- We showed how to filter, normalize, and group data by attributes, with all processing steps referencing schema IDs for maximum reproducibility.

**You can now adapt this workflow to explore, process, and visualize other datasets with a Croissant schema!**